# Module 2: Improving Deep Neural Networks

## Learning Objectives
- Master hyperparameter tuning techniques
- Implement various regularization methods
- Understand and apply optimization algorithms
- Learn batch normalization and its benefits

## 2.1 Hyperparameter Tuning

Key hyperparameters in deep learning:
- Learning rate
- Batch size
- Number of layers/neurons
- Activation functions
- Regularization parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Generate synthetic data
X, y = make_classification(n_samples=2000, n_features=20, n_informative=15, 
                           n_redundant=5, n_clusters_per_class=1, random_state=42)
y = y.reshape(-1, 1)

# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

## 2.2 Learning Rate Scheduling

In [ ]:
def create_model(learning_rate=0.001, hidden_units=64, dropout_rate=0.2):
    """Create a simple neural network with configurable parameters"""
    model = keras.Sequential([
        layers.Dense(hidden_units, activation='relu', input_shape=(X_train.shape[1],)),
        layers.Dropout(dropout_rate),
        layers.Dense(hidden_units//2, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1, activation='sigmoid')
    ])
    
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

# Test different learning rates
learning_rates = [0.001, 0.01, 0.1, 0.0001]
histories = {}

for lr in learning_rates:
    print(f"\nTraining with learning rate: {lr}")
    model = create_model(learning_rate=lr)
    
    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )
    
    histories[lr] = history
    
    # Evaluate on test set
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test accuracy: {test_acc:.4f}")

# Plot training curves
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
for lr in learning_rates:
    plt.plot(histories[lr].history['loss'], label=f'LR={lr}')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
for lr in learning_rates:
    plt.plot(histories[lr].history['accuracy'], label=f'LR={lr}')
plt.title('Training Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
for lr in learning_rates:
    plt.plot(histories[lr].history['val_loss'], label=f'LR={lr}')
plt.title('Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 2.3 Regularization Techniques

In [ ]:
# L1 and L2 Regularization
def create_regularized_model(l1_ratio=0.0, l2_ratio=0.0, dropout_rate=0.0):
    """Create model with different regularization techniques"""
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],),
                   kernel_regularizer=keras.regularizers.l1_l2(l1=l1_ratio, l2=l2_ratio)),
        layers.Dropout(dropout_rate),
        layers.Dense(64, activation='relu',
                   kernel_regularizer=keras.regularizers.l1_l2(l1=l1_ratio, l2=l2_ratio)),
        layers.Dropout(dropout_rate),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Test different regularization strategies
regularization_configs = [
    {'name': 'No Regularization', 'l1': 0.0, 'l2': 0.0, 'dropout': 0.0},
    {'name': 'L2 Only', 'l1': 0.0, 'l2': 0.01, 'dropout': 0.0},
    {'name': 'L1 Only', 'l1': 0.01, 'l2': 0.0, 'dropout': 0.0},
    {'name': 'L1 + L2', 'l1': 0.01, 'l2': 0.01, 'dropout': 0.0},
    {'name': 'Dropout Only', 'l1': 0.0, 'l2': 0.0, 'dropout': 0.3},
    {'name': 'All Regularization', 'l1': 0.01, 'l2': 0.01, 'dropout': 0.3}
]

reg_results = {}

for config in regularization_configs:
    print(f"\nTraining with {config['name']}")
    model = create_regularized_model(
        l1_ratio=config['l1'], 
        l2_ratio=config['l2'], 
        dropout_rate=config['dropout']
    )
    
    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )
    
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    reg_results[config['name']] = {
        'history': history,
        'test_acc': test_acc,
        'test_loss': test_loss
    }
    
    print(f"Test accuracy: {test_acc:.4f}")

# Plot results
plt.figure(figsize=(15, 10))

# Training loss comparison
plt.subplot(2, 2, 1)
for name, result in reg_results.items():
    plt.plot(result['history'].history['loss'], label=name)
plt.title('Training Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Validation loss comparison
plt.subplot(2, 2, 2)
for name, result in reg_results.items():
    plt.plot(result['history'].history['val_loss'], label=name)
plt.title('Validation Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Test accuracy comparison
plt.subplot(2, 2, 3)
names = list(reg_results.keys())
test_accs = [reg_results[name]['test_acc'] for name in names]
bars = plt.bar(names, test_accs)
plt.title('Test Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)

# Add value labels on bars
for bar, acc in zip(bars, test_accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{acc:.3f}', ha='center', va='bottom')

# Overfitting analysis (train vs val accuracy gap)
plt.subplot(2, 2, 4)
overfitting_gaps = []
for name, result in reg_results.items():
    final_train_acc = result['history'].history['accuracy'][-1]
    final_val_acc = result['history'].history['val_accuracy'][-1]
    gap = final_train_acc - final_val_acc
    overfitting_gaps.append(gap)

bars = plt.bar(names, overfitting_gaps)
plt.title('Overfitting Gap (Train - Val Accuracy)')
plt.ylabel('Accuracy Gap')
plt.xticks(rotation=45)
plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)

for bar, gap in zip(bars, overfitting_gaps):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{gap:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 2.4 Optimization Algorithms

In [ ]:
# Compare different optimizers
def create_model_for_optimizer(optimizer_name):
    """Create model for testing different optimizers"""
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    
    if optimizer_name == 'SGD':
        optimizer = keras.optimizers.SGD(learning_rate=0.01)
    elif optimizer_name == 'SGD_momentum':
        optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
    elif optimizer_name == 'RMSprop':
        optimizer = keras.optimizers.RMSprop(learning_rate=0.001)
    elif optimizer_name == 'Adam':
        optimizer = keras.optimizers.Adam(learning_rate=0.001)
    elif optimizer_name == 'Nadam':
        optimizer = keras.optimizers.Nadam(learning_rate=0.001)
    else:
        optimizer = 'adam'
    
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Test different optimizers
optimizers = ['SGD', 'SGD_momentum', 'RMSprop', 'Adam', 'Nadam']
optimizer_results = {}

for opt_name in optimizers:
    print(f"\nTraining with {opt_name}")
    model = create_model_for_optimizer(opt_name)
    
    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )
    
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    optimizer_results[opt_name] = {
        'history': history,
        'test_acc': test_acc,
        'test_loss': test_loss
    }
    
    print(f"Test accuracy: {test_acc:.4f}")

# Plot optimizer comparison
plt.figure(figsize=(15, 8))

plt.subplot(2, 3, 1)
for opt_name, result in optimizer_results.items():
    plt.plot(result['history'].history['loss'], label=opt_name)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(2, 3, 2)
for opt_name, result in optimizer_results.items():
    plt.plot(result['history'].history['val_loss'], label=opt_name)
plt.title('Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(2, 3, 3)
for opt_name, result in optimizer_results.items():
    plt.plot(result['history'].history['accuracy'], label=opt_name)
plt.title('Training Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(2, 3, 4)
for opt_name, result in optimizer_results.items():
    plt.plot(result['history'].history['val_accuracy'], label=opt_name)
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Final performance comparison
plt.subplot(2, 3, 5)
names = list(optimizer_results.keys())
test_accs = [optimizer_results[name]['test_acc'] for name in names]
bars = plt.bar(names, test_accs)
plt.title('Final Test Accuracy')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)

for bar, acc in zip(bars, test_accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{acc:.3f}', ha='center', va='bottom')

# Convergence speed (epochs to reach 90% of final accuracy)
plt.subplot(2, 3, 6)
convergence_epochs = []
for opt_name, result in optimizer_results.items():
    target_acc = 0.9 * result['test_acc']
    val_accs = result['history'].history['val_accuracy']
    convergence_epoch = None
    for epoch, acc in enumerate(val_accs):
        if acc >= target_acc:
            convergence_epoch = epoch
            break
    if convergence_epoch is None:
        convergence_epoch = len(val_accs)
    convergence_epochs.append(convergence_epoch)

bars = plt.bar(names, convergence_epochs)
plt.title('Convergence Speed (Epochs to 90% Accuracy)')
plt.ylabel('Epochs')
plt.xticks(rotation=45)

for bar, epoch in zip(bars, convergence_epochs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{epoch}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 2.5 Batch Normalization

In [ ]:
# Compare models with and without batch normalization
def create_model_with_bn(use_bn=True):
    """Create model with optional batch normalization"""
    model = keras.Sequential()
    
    # First layer
    model.add(layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
    if use_bn:
        model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.2))
    
    # Second layer
    model.add(layers.Dense(64, activation='relu'))
    if use_bn:
        model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.2))
    
    # Output layer
    model.add(layers.Dense(1, activation='sigmoid'))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Train models with and without batch normalization
print("Training without Batch Normalization:")
model_no_bn = create_model_with_bn(use_bn=False)
history_no_bn = model_no_bn.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

test_loss_no_bn, test_acc_no_bn = model_no_bn.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy without BN: {test_acc_no_bn:.4f}")

print("\nTraining with Batch Normalization:")
model_with_bn = create_model_with_bn(use_bn=True)
history_with_bn = model_with_bn.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

test_loss_bn, test_acc_bn = model_with_bn.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy with BN: {test_acc_bn:.4f}")

# Plot comparison
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.plot(history_no_bn.history['loss'], label='Without BN')
plt.plot(history_with_bn.history['loss'], label='With BN')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(2, 3, 2)
plt.plot(history_no_bn.history['val_loss'], label='Without BN')
plt.plot(history_with_bn.history['val_loss'], label='With BN')
plt.title('Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(2, 3, 3)
plt.plot(history_no_bn.history['accuracy'], label='Without BN')
plt.plot(history_with_bn.history['accuracy'], label='With BN')
plt.title('Training Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(2, 3, 4)
plt.plot(history_no_bn.history['val_accuracy'], label='Without BN')
plt.plot(history_with_bn.history['val_accuracy'], label='With BN')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Final comparison
plt.subplot(2, 3, 5)
methods = ['Without BN', 'With BN']
test_accs = [test_acc_no_bn, test_acc_bn]
bars = plt.bar(methods, test_accs, color=['red', 'green'])
plt.title('Final Test Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim(0, 1)

for bar, acc in zip(bars, test_accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.3f}', ha='center', va='bottom')

# Training stability (variance in loss)
plt.subplot(2, 3, 6)
loss_var_no_bn = np.var(history_no_bn.history['loss'][-20:])  # Last 20 epochs
loss_var_bn = np.var(history_with_bn.history['loss'][-20:])

bars = plt.bar(['Without BN', 'With BN'], [loss_var_no_bn, loss_var_bn], color=['red', 'green'])
plt.title('Training Stability (Loss Variance)')
plt.ylabel('Variance')

for bar, var in zip(bars, [loss_var_no_bn, loss_var_bn]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + var*0.01,
             f'{var:.6f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 2.6 Early Stopping and Model Checkpointing

In [ ]:
# Implement early stopping and model checkpointing
def create_model_for_callbacks():
    """Create model for callback testing"""
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Define callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

model_checkpoint = keras.callbacks.ModelCheckpoint(
    'best_model.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# Train with callbacks
model_callbacks = create_model_for_callbacks()

print("Training with callbacks:")
history_callbacks = model_callbacks.fit(
    X_train, y_train,
    epochs=200,  # Large number, early stopping will prevent overtraining
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr, model_checkpoint],
    verbose=1
)

# Load best model
best_model = keras.models.load_model('best_model.h5')
test_loss_best, test_acc_best = best_model.evaluate(X_test, y_test, verbose=0)

print(f"\nBest model test accuracy: {test_acc_best:.4f}")
print(f"Training stopped at epoch: {len(history_callbacks.history['loss'])}")

# Plot training history with callbacks
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history_callbacks.history['loss'], label='Training Loss')
plt.plot(history_callbacks.history['val_loss'], label='Validation Loss')
plt.axvline(x=len(history_callbacks.history['loss']) - early_stopping.patience, 
           color='red', linestyle='--', label='Early Stopping Point')
plt.title('Loss with Early Stopping')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(history_callbacks.history['accuracy'], label='Training Accuracy')
plt.plot(history_callbacks.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy with Early Stopping')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
# Plot learning rate changes
if 'lr' in history_callbacks.history:
    plt.plot(history_callbacks.history['lr'], label='Learning Rate')
    plt.title('Learning Rate Schedule')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.yscale('log')
    plt.legend()
    plt.grid(True)
else:
    plt.text(0.5, 0.5, 'Learning rate not tracked', ha='center', va='center')
    plt.title('Learning Rate Schedule')

plt.tight_layout()
plt.show()

## 2.7 Key Takeaways

### Hyperparameter Tuning
- **Learning rate** is the most critical hyperparameter
- **Batch size** affects training speed and generalization
- **Network architecture** should be adapted to problem complexity

### Regularization Techniques
- **L1 regularization** encourages sparse weights
- **L2 regularization** prevents large weights
- **Dropout** randomly deactivates neurons during training
- **Early stopping** prevents overfitting

### Optimization Algorithms
- **SGD**: Simple but can be slow
- **SGD with momentum**: Faster convergence
- **Adam**: Adaptive learning rates, good default choice
- **RMSprop**: Good for RNNs

### Batch Normalization
- Stabilizes training
- Allows higher learning rates
- Reduces sensitivity to initialization
- Acts as a regularizer

## Exercises

1. **Hyperparameter Search**: Implement a grid search or random search for optimal hyperparameters
2. **Learning Rate Schedules**: Try different learning rate schedules (step decay, cosine annealing)
3. **Advanced Regularization**: Implement dropout variants (Monte Carlo dropout)
4. **Optimizer Comparison**: Test optimizers on different datasets
5. **Batch Normalization Placement**: Experiment with different BN placements in the network